# CBIS-DDSM DeepProbLog Training

This notebook trains the neuro-symbolic pipeline:

`Image → neural concept nets (shape / margin / calc) → DeepProbLog rules → malignant(X)`

It expects patient-disjoint CSVs from `cbis_ddsm_data_prep.ipynb`. If those files are missing, it will rebuild splits from the four official description CSVs.

**Important**
- `SMOKE_TEST=True` uses a tiny subset and few epochs. Do not report these numbers in the paper.
- The Prolog rules are a **provisional computational program**, not official BI-RADS diagnostic criteria.
- Concept vocabularies are coarse mappings of CBIS-DDSM descriptors.

In [ ]:
# Install DeepProbLog and deps (Kaggle / Colab).
# pysdd is required by DeepProbLog for exact inference.
%pip install -q deepproblog scikit-learn pandas pillow

In [ ]:
import json
import os
import random
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from PIL import Image
from sklearn.model_selection import GroupShuffleSplit
from torchvision import transforms

print("torch:", torch.__version__)
print("cuda:", torch.cuda.is_available())

In [ ]:
# --- Configuration ---
SMOKE_TEST = True
SMOKE_N_SAMPLES = 24          # rows per split in smoke mode
EPOCHS = 1 if SMOKE_TEST else 5
BATCH_SIZE = 2 if SMOKE_TEST else 4
IMAGE_SIZE = 64 if SMOKE_TEST else 128
LR = 1e-3
RANDOM_SEED = 42
VAL_SIZE = 0.15
USE_SYNTHETIC_IF_MISSING = True  # needed locally if JPEGs are not present

# Image dataset (CBIS-DDSM JPEGs)
KAGGLE_INPUT = Path("/kaggle/input/cbis-ddsm-breast-cancer-image-dataset")
KAGGLE_WORKING = Path("/kaggle/working")
LOCAL_DATA = Path("../data")
LOCAL_PROCESSED = LOCAL_DATA / "processed"

# Uploaded smoke-test dataset: processed CSVs + breast_diagnosis.pl
# Attach this dataset in Kaggle: Add Input → smoke-test-problog
SMOKE_DATASET = Path("/kaggle/input/smoke-test-problog")
PROCESSED_KAGGLE_CANDIDATES = [
    SMOKE_DATASET,
    SMOKE_DATASET / "processed",
    Path("/kaggle/input/smoketestproblog"),
    Path("/kaggle/input/smoke-test-problog/processed"),
]

PROGRAM_CANDIDATES = [
    SMOKE_DATASET / "breast_diagnosis.pl",
    SMOKE_DATASET / "programs" / "breast_diagnosis.pl",
    Path("programs/breast_diagnosis.pl"),
    Path("../notebooks/programs/breast_diagnosis.pl"),
    Path("/kaggle/working/breast_diagnosis.pl"),
]

random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)
torch.manual_seed(RANDOM_SEED)

if SMOKE_TEST:
    print("=" * 64)
    print("SMOKE_TEST=True — DeepProbLog training is a pipeline check only.")
    print("Do not report accuracy / F1 / AUC from this run in the paper.")
    print("=" * 64)

print("Expecting processed CSVs + program from:", SMOKE_DATASET)


## Load patient-disjoint tables

In [ ]:
CSV_FILENAMES = {
    "mass_train": "mass_case_description_train_set.csv",
    "mass_test": "mass_case_description_test_set.csv",
    "calc_train": "calc_case_description_train_set.csv",
    "calc_test": "calc_case_description_test_set.csv",
}


def find_csv_paths(root, filenames):
    root = Path(root)
    found = {}
    remaining = set(filenames.values())
    for dirpath, dirnames, files in os.walk(root):
        dirnames[:] = [d for d in dirnames if d.lower() != "jpeg"]
        for fname in files:
            if fname in remaining:
                key = next(k for k, v in filenames.items() if v == fname)
                found[key] = Path(dirpath) / fname
                remaining.discard(fname)
        if not remaining:
            break
    missing = [k for k in filenames if k not in found]
    if missing:
        raise FileNotFoundError(f"Missing {missing} under {root}")
    return found


def normalize_columns(df):
    out = df.copy()
    out.columns = [c.strip() for c in out.columns]
    if "breast density" in out.columns:
        out = out.rename(columns={"breast density": "breast_density"})
    for col in out.columns:
        if out[col].dtype == object:
            out[col] = out[col].astype(str).str.strip().replace({"nan": np.nan, "None": np.nan, "": np.nan})
    return out


def rebuild_splits_from_raw(data_root):
    paths = find_csv_paths(data_root, CSV_FILENAMES)
    mass_train = normalize_columns(pd.read_csv(paths["mass_train"]))
    mass_test = normalize_columns(pd.read_csv(paths["mass_test"]))
    calc_train = normalize_columns(pd.read_csv(paths["calc_train"]))
    calc_test = normalize_columns(pd.read_csv(paths["calc_test"]))
    mass_train["abnormality_kind"] = "mass"
    mass_test["abnormality_kind"] = "mass"
    calc_train["abnormality_kind"] = "calcification"
    calc_test["abnormality_kind"] = "calcification"
    train_all = pd.concat([mass_train, calc_train], ignore_index=True)
    test_all = pd.concat([mass_test, calc_test], ignore_index=True)
    gss = GroupShuffleSplit(n_splits=1, test_size=VAL_SIZE, random_state=RANDOM_SEED)
    tr_idx, va_idx = next(gss.split(train_all, groups=train_all["patient_id"]))
    return (
        train_all.iloc[tr_idx].reset_index(drop=True),
        train_all.iloc[va_idx].reset_index(drop=True),
        test_all.reset_index(drop=True),
    )


def locate_processed():
    """Find folder containing cbis_train/val/test.csv.

    Prefer the uploaded smoke-test-problog dataset, then scan /kaggle/input.
    """
    candidates = []
    # 1) Uploaded smoke-test dataset (highest priority)
    candidates.extend(PROCESSED_KAGGLE_CANDIDATES)
    # 2) Working / local fallbacks
    if KAGGLE_WORKING.is_dir():
        candidates.append(KAGGLE_WORKING / "processed")
    candidates.append(Path("../notebooks/processed"))
    candidates.append(LOCAL_PROCESSED)
    # 3) Auto-scan all Kaggle inputs
    kaggle_input = Path("/kaggle/input")
    if kaggle_input.is_dir():
        for train_csv in kaggle_input.rglob("cbis_train.csv"):
            candidates.append(train_csv.parent)

    seen = set()
    for folder in candidates:
        folder = Path(folder)
        key = str(folder)
        if key in seen:
            continue
        seen.add(key)
        needed = [folder / f"cbis_{s}.csv" for s in ("train", "val", "test")]
        if all(p.is_file() for p in needed):
            return folder
    return None


def locate_program():
    """Find breast_diagnosis.pl (prefer smoke-test-problog dataset)."""
    candidates = list(PROGRAM_CANDIDATES)
    kaggle_input = Path("/kaggle/input")
    if kaggle_input.is_dir():
        candidates.extend(kaggle_input.rglob("breast_diagnosis.pl"))
        candidates.extend(kaggle_input.rglob("*.pl"))
    for path in candidates:
        path = Path(path)
        if path.is_file() and path.name.lower() in {"breast_diagnosis.pl", "breast-diagnosis.pl"}:
            return path
    # Any .pl under smoke dataset as last resort
    for path in candidates:
        path = Path(path)
        if path.is_file() and path.suffix.lower() == ".pl":
            return path
    return None


processed_dir = locate_processed()
program_file = locate_program()

if processed_dir is not None:
    print(f"Loading processed splits from {processed_dir}")
    df_train = pd.read_csv(processed_dir / "cbis_train.csv")
    df_val = pd.read_csv(processed_dir / "cbis_val.csv")
    df_test = pd.read_csv(processed_dir / "cbis_test.csv")
else:
    data_root = KAGGLE_INPUT if KAGGLE_INPUT.is_dir() else LOCAL_DATA
    print(f"Processed CSVs not found; rebuilding splits from {data_root}")
    df_train, df_val, df_test = rebuild_splits_from_raw(data_root)

print("Program file:", program_file if program_file else "NOT FOUND (will write fallback later)")

for name, df in (("train", df_train), ("val", df_val), ("test", df_test)):
    print(f"{name:5s}: {len(df):>5} rows | {df['patient_id'].nunique():>4} patients")


## Concept labels and image tensors

Coarse concept inventories used here:
- **shape:** `regular` (round/oval/lobulated), `irregular`, or `unknown`
- **margin:** `circumscribed`, `spiculated`, `other`, or `unknown`
- **calc:** `present` for calcification cases, `absent` otherwise

These are dataset-derived groupings, not a complete BI-RADS lexicon.

In [ ]:
def tokenize_descriptor(value):
    if pd.isna(value):
        return []
    text = str(value).strip().upper()
    if text in {"", "NAN", "NONE"}:
        return []
    for sep in ("-", "/", "_", " "):
        text = text.replace(sep, " ")
    return [t for t in text.split() if t]


def map_shape(value):
    tokens = tokenize_descriptor(value)
    if not tokens:
        return "unknown"
    if any(t in {"IRREGULAR", "ARCHITECTURAL", "DISTORTION"} for t in tokens):
        return "irregular"
    if any(t in {"ROUND", "OVAL", "LOBULATED"} for t in tokens):
        return "regular"
    return "unknown"


def map_margin(value):
    tokens = tokenize_descriptor(value)
    if not tokens:
        return "unknown"
    if "SPICULATED" in tokens:
        return "spiculated"
    if "CIRCUMSCRIBED" in tokens:
        return "circumscribed"
    return "other"


def map_calc(row):
    kind = str(row.get("abnormality_kind", "")).lower()
    if kind != "calcification":
        return "absent"
    text = str(row.get("calc type", "")).upper().replace("-", " ").replace("/", " ").replace("_", " ")
    tokens = [t for t in text.split() if t]
    if any(t in {"PLEOMORPHIC", "AMORPHOUS"} for t in tokens) or (
        "FINE" in tokens and "LINEAR" in tokens
    ) or "BRANCHING" in tokens:
        return "suspicious"
    return "typically_benign"


def map_malignant(pathology):
    return 1.0 if str(pathology).strip().upper() == "MALIGNANT" else 0.0


def add_concepts(df):
    out = df.copy()
    shape_col = "mass shape" if "mass shape" in out.columns else None
    margin_col = "mass margins" if "mass margins" in out.columns else None
    out["shape_concept"] = out[shape_col].map(map_shape) if shape_col else "unknown"
    out["margin_concept"] = out[margin_col].map(map_margin) if margin_col else "unknown"
    out["calc_concept"] = out.apply(map_calc, axis=1)
    out["malignant"] = out["pathology"].map(map_malignant)
    return out


df_train = add_concepts(df_train)
df_val = add_concepts(df_val)
df_test = add_concepts(df_test)
display(df_train[["patient_id", "abnormality_kind", "shape_concept", "margin_concept", "calc_concept", "malignant"]].head())


In [ ]:
def find_jpeg_root():
    candidates = [
        KAGGLE_INPUT / "jpeg",
        LOCAL_DATA / "jpeg",
    ]
    if KAGGLE_INPUT.is_dir():
        candidates.extend(p for p in KAGGLE_INPUT.rglob("jpeg") if p.is_dir())
    for p in candidates:
        if p.is_dir():
            return p
    return None


JPEG_ROOT = find_jpeg_root()
print("JPEG root:", JPEG_ROOT)


def clean_relpath(value):
    if pd.isna(value):
        return None
    text = str(value).strip().strip('"').split("\n")[0].strip()
    return text or None


def resolve_jpeg(row):
    if JPEG_ROOT is None:
        return None
    for col in ("cropped image file path", "image file path"):
        rel = clean_relpath(row.get(col))
        if not rel:
            continue
        parts = Path(rel.replace("\\", "/")).parts
        if not parts:
            continue
        folder = JPEG_ROOT.joinpath(*parts[:-1]) if len(parts) > 1 else JPEG_ROOT / parts[0]
        if folder.is_dir():
            jpgs = sorted(folder.glob("*.jpg")) + sorted(folder.glob("*.png"))
            if jpgs:
                return jpgs[0]
        case_dir = JPEG_ROOT / parts[0]
        if case_dir.is_dir():
            jpgs = sorted(case_dir.rglob("*.jpg")) + sorted(case_dir.rglob("*.png"))
            if jpgs:
                return jpgs[0]
    return None


tfm = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.ToTensor(),
])


def load_tensor(row):
    path = resolve_jpeg(row)
    if path is not None and path.is_file():
        with Image.open(path) as img:
            img = img.convert("L")
            return tfm(img), str(path), False
    if not USE_SYNTHETIC_IF_MISSING:
        raise FileNotFoundError(f"No JPEG for patient {row.get('patient_id')}")
    synth = torch.rand(1, IMAGE_SIZE, IMAGE_SIZE)
    return synth, None, True


def materialize_split(df, n_limit=None):
    if n_limit is not None:
        df = df.head(n_limit).copy()
    tensors = []
    records = []
    n_synth = 0
    for _, row in df.iterrows():
        tensor, path, synthetic = load_tensor(row)
        tensors.append(tensor)
        rec = row.to_dict()
        rec["resolved_path"] = path
        rec["synthetic"] = synthetic
        records.append(rec)
        n_synth += int(synthetic)
    print(f"  loaded {len(tensors)} tensors ({n_synth} synthetic)")
    return tensors, pd.DataFrame(records)


limit = SMOKE_N_SAMPLES if SMOKE_TEST else None
print("Materializing train")
train_tensors, df_train = materialize_split(df_train, limit)
print("Materializing val")
val_tensors, df_val = materialize_split(df_val, limit)
print("Materializing test")
test_tensors, df_test = materialize_split(df_test, limit)

if all(df["synthetic"].all() for df in (df_train, df_val, df_test)):
    print("WARNING: all images are synthetic. Training will run, but results are not meaningful.")

## Neural concept extractors + DeepProbLog dataset

In [ ]:
class ConceptCNN(nn.Module):
    """Small grayscale CNN used as a DeepProbLog neural predicate."""

    def __init__(self, n_classes):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(1, 16, 3, stride=2, padding=1),
            nn.ReLU(inplace=True),
            nn.Conv2d(16, 32, 3, stride=2, padding=1),
            nn.ReLU(inplace=True),
            nn.Conv2d(32, 64, 3, stride=2, padding=1),
            nn.ReLU(inplace=True),
            nn.AdaptiveAvgPool2d(4),
        )
        self.classifier = nn.Linear(64 * 4 * 4, n_classes)
        self.softmax = nn.Softmax(dim=1)

    def forward(self, x):
        feats = self.net(x).flatten(1)
        return self.softmax(self.classifier(feats))


shape_net = ConceptCNN(3)    # regular, irregular, unknown
margin_net = ConceptCNN(4)   # circumscribed, spiculated, other, unknown
calc_net = ConceptCNN(3)     # absent, typically_benign, suspicious

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
shape_net.to(device)
margin_net.to(device)
calc_net.to(device)
print("device:", device)


In [ ]:
from problog.logic import Term, Constant
from deepproblog.dataset import Dataset, DataLoader, QueryDataset
from deepproblog.query import Query


class TensorSource:
    def __init__(self, tensors):
        self.tensors = tensors

    def __getitem__(self, item):
        idx = int(item[0])
        return self.tensors[idx]


class BreastQueryDataset(Dataset):
    """Diagnostic queries: malignant(tensor(split, i)) with target probability 0/1."""

    def __init__(self, split_name, df):
        self.split_name = split_name
        self.df = df.reset_index(drop=True)

    def __len__(self):
        return len(self.df)

    def _tensor_term(self, i):
        return Term("tensor", Term(self.split_name, Constant(i)))

    def to_query(self, i):
        row = self.df.iloc[i]
        img = self._tensor_term(i)
        return Query(Term("malignant", img), p=float(row["malignant"]))

    def concept_queries(self):
        queries = []
        for i, row in self.df.iterrows():
            img = self._tensor_term(i)
            queries.append(Query(Term("shape", img, Term(row["shape_concept"])), p=1.0))
            queries.append(Query(Term("margin", img, Term(row["margin_concept"])), p=1.0))
            queries.append(Query(Term("calc", img, Term(row["calc_concept"])), p=1.0))
        return queries


train_set = BreastQueryDataset("train", df_train)
val_set = BreastQueryDataset("val", df_val)
test_set = BreastQueryDataset("test", df_test)
# Supervise concepts and the diagnostic query together.
train_queries = QueryDataset(train_set.to_queries() + train_set.concept_queries())
print("example malignant query:", train_set.to_query(0))
print("n diagnostic queries:", len(train_set), "| n train queries with concepts:", len(train_queries))


## Train DeepProbLog

The model file uses neural predicates `shape_net`, `margin_net`, and `calc_net`. Gradients flow through probabilistic inference into the shared encoder.

In [ ]:
from deepproblog.engines import ExactEngine
from deepproblog.model import Model
from deepproblog.network import Network
from deepproblog.train import train_model

PROGRAM_FALLBACK = """
nn(shape_net, [X], Y, [regular, irregular, unknown]) :: shape(X, Y).
nn(margin_net, [X], Y, [circumscribed, spiculated, other, unknown]) :: margin(X, Y).
nn(calc_net, [X], Y, [absent, typically_benign, suspicious]) :: calc(X, Y).

malignant(X) :- shape(X, irregular).
malignant(X) :- margin(X, spiculated).
malignant(X) :- calc(X, suspicious).
""".strip()

# Prefer program discovered earlier from smoke-test-problog (or scan again)
program_path = program_file if "program_file" in globals() and program_file is not None else locate_program()
if program_path is None:
    program_path = Path("/kaggle/working/breast_diagnosis.pl") if Path("/kaggle/working").is_dir() else Path("breast_diagnosis.pl")
    program_path.write_text(PROGRAM_FALLBACK)
    print("Wrote fallback program to", program_path)
print("Using program:", program_path.resolve())
print(program_path.read_text())

shape_wrapper = Network(shape_net, "shape_net", batching=True)
margin_wrapper = Network(margin_net, "margin_net", batching=True)
calc_wrapper = Network(calc_net, "calc_net", batching=True)
shape_wrapper.optimizer = torch.optim.Adam(shape_net.parameters(), lr=LR)
margin_wrapper.optimizer = torch.optim.Adam(margin_net.parameters(), lr=LR)
calc_wrapper.optimizer = torch.optim.Adam(calc_net.parameters(), lr=LR)

model = Model(str(program_path), [shape_wrapper, margin_wrapper, calc_wrapper])
model.set_engine(ExactEngine(model), cache=False)
model.add_tensor_source("train", TensorSource(train_tensors))
model.add_tensor_source("val", TensorSource(val_tensors))
model.add_tensor_source("test", TensorSource(test_tensors))

loader = DataLoader(train_queries, BATCH_SIZE, shuffle=True, seed=RANDOM_SEED)
print(f"Training DeepProbLog for {EPOCHS} epoch(s), batch_size={BATCH_SIZE}, n_queries={len(train_queries)}")
train_log = train_model(model, loader, EPOCHS, log_iter=max(1, len(loader)))
print("Training finished.")


## Evaluation and reasoning trace

Metrics below are for the current run only. If `SMOKE_TEST=True`, they are **not** paper results.

In [ ]:
def extract_prob(result):
    """DeepProbLog model.solve returns a Result object with .result dict[Term -> prob]."""
    if result is None:
        return 0.0
    # Result object
    if hasattr(result, "result"):
        mapping = result.result
    elif isinstance(result, dict):
        mapping = result
    else:
        return 0.0
    if not mapping:
        return 0.0
    vals = []
    for v in mapping.values():
        if hasattr(v, "detach"):
            vals.append(float(v.detach().cpu().item() if v.numel() == 1 else v.detach().cpu().mean()))
        else:
            vals.append(float(v))
    return float(sum(vals) / len(vals))


def malignant_probability(split_name, i):
    q = Query(Term("malignant", Term("tensor", Term(split_name, Constant(i)))))
    solved = model.solve([q])
    if not solved:
        return 0.0
    return extract_prob(solved[0])


def evaluate_split(name, dataset, df):
    y_true = []
    y_prob = []
    for i in range(len(dataset)):
        y_true.append(int(df.iloc[i]["malignant"]))
        y_prob.append(malignant_probability(name, i))
    y_true = np.array(y_true)
    y_prob = np.array(y_prob)
    y_pred = (y_prob >= 0.5).astype(int)
    tp = int(((y_pred == 1) & (y_true == 1)).sum())
    tn = int(((y_pred == 0) & (y_true == 0)).sum())
    fp = int(((y_pred == 1) & (y_true == 0)).sum())
    fn = int(((y_pred == 0) & (y_true == 1)).sum())
    acc = (tp + tn) / max(len(y_true), 1)
    sens = tp / max(tp + fn, 1)
    spec = tn / max(tn + fp, 1)
    return {
        "split": name,
        "n": len(y_true),
        "accuracy": acc,
        "sensitivity": sens,
        "specificity": spec,
        "tp": tp, "tn": tn, "fp": fp, "fn": fn,
    }, y_prob


metrics_val, val_probs = evaluate_split("val", val_set, df_val)
metrics_test, test_probs = evaluate_split("test", test_set, df_test)
metrics_df = pd.DataFrame([metrics_val, metrics_test])
display(metrics_df)
if SMOKE_TEST:
    print("SMOKE_TEST=True: do not copy these numbers into the paper.")


In [ ]:
def inspect_case(split_name, df, i):
    row = df.iloc[i]
    img = Term("tensor", Term(split_name, Constant(i)))
    queries = {
        "malignant": Query(Term("malignant", img)),
        "shape_irregular": Query(Term("shape", img, Term("irregular"))),
        "margin_spiculated": Query(Term("margin", img, Term("spiculated"))),
        "calc_suspicious": Query(Term("calc", img, Term("suspicious"))),
    }
    print(f"\n[{split_name} #{i}] patient={row['patient_id']} pathology={row['pathology']} kind={row['abnormality_kind']}")
    print(f"  gold concepts: shape={row['shape_concept']} margin={row['margin_concept']} calc={row['calc_concept']}")
    for name, q in queries.items():
        solved = model.solve([q])
        p = extract_prob(solved[0]) if solved else 0.0
        print(f"  P({name}) = {p:.3f}")


n_show = min(3, len(val_set))
print("Example reasoning traces (validation):")
for i in range(n_show):
    inspect_case("val", df_val, i)


In [ ]:
out_dir = KAGGLE_WORKING / "processed" if KAGGLE_WORKING.is_dir() else LOCAL_PROCESSED
out_dir.mkdir(parents=True, exist_ok=True)
snapshot = out_dir / "deepproblog_smoke.pth" if SMOKE_TEST else out_dir / "deepproblog_model.pth"
model.save_state(str(snapshot))
metrics_path = out_dir / "deepproblog_metrics.json"
payload = {
    "smoke_test": SMOKE_TEST,
    "epochs": EPOCHS,
    "image_size": IMAGE_SIZE,
    "n_train": len(train_set),
    "n_val": len(val_set),
    "n_test": len(test_set),
    "metrics_val": metrics_val,
    "metrics_test": metrics_test,
    "note": "Do not report smoke-test numbers in the paper. Rules are provisional, not official BI-RADS."
}
metrics_path.write_text(json.dumps(payload, indent=2))
print("Saved:", snapshot)
print("Saved:", metrics_path)